In [ ]:
# @title Set Up
import sys

assert sys.version_info >= (3, 7)

from packaging import version
import sklearn

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

# Ensemble Learning

### Create Moon data

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=250, noise=0.2, random_state=42)

plt.scatter(X_moons[:,0],X_moons[:,1],c=y_moons)

### Single regularized decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_clf = DecisionTreeClassifier(min_samples_leaf=5, max_depth=5, splitter="random", random_state=42)

tree_clf.fit(X_moons, y_moons)

### Voting among many decision trees (we use splitter=random to ensure each tree behave slightly different from the others)

In [ ]:
from sklearn.ensemble import VotingClassifier

tree_clf1 = DecisionTreeClassifier(min_samples_leaf=5, max_depth=5, splitter="random", random_state=1)
tree_clf2 = DecisionTreeClassifier(min_samples_leaf=5, max_depth=5, splitter="random", random_state=2)
tree_clf3 = DecisionTreeClassifier(min_samples_leaf=5, max_depth=5, splitter="random", random_state=3)
tree_clf4 = DecisionTreeClassifier(min_samples_leaf=5, max_depth=5, splitter="random", random_state=4)
tree_clf5 = DecisionTreeClassifier(min_samples_leaf=5, max_depth=5, splitter="random", random_state=5)

vote_clf = VotingClassifier(estimators=[
    ('tree1',tree_clf1),
    ('tree2',tree_clf2),
    ('tree3',tree_clf3),
    ('tree4',tree_clf4),
    ('tree5',tree_clf5)],voting='hard')

vote_clf.fit(X_moons, y_moons)

In [ ]:
# @title extra code – for plotting

def plot_decision_boundary(clf, X, y, axes, cmap):
    x1, x2 = np.meshgrid(np.linspace(axes[0], axes[1], 100),
                         np.linspace(axes[2], axes[3], 100))
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)

    plt.contourf(x1, x2, y_pred, alpha=0.3, cmap=cmap)
    plt.contour(x1, x2, y_pred, cmap="Greys", alpha=0.8)
    colors = {"Wistia": ["#78785c", "#c47b27"], "Pastel1": ["red", "blue"]}
    markers = ("o", "^")
    for idx in (0, 1):
        plt.plot(X[:, 0][y == idx], X[:, 1][y == idx],
                 color=colors[cmap][idx], marker=markers[idx], linestyle="none")
    plt.axis(axes)
    plt.xlabel(r"$x_1$")
    plt.ylabel(r"$x_2$", rotation=0)

fig, axes = plt.subplots(ncols=2, figsize=(10, 4), sharey=True)
plt.sca(axes[0])
plot_decision_boundary(tree_clf, X_moons, y_moons,
                       axes=[-1.5, 2.4, -1, 1.5], cmap="Wistia")
plt.title("Single Tree")
plt.sca(axes[1])
plot_decision_boundary(vote_clf, X_moons, y_moons,
                       axes=[-1.5, 2.4, -1, 1.5], cmap="Wistia")
plt.title("Voting")
plt.ylabel("")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

#calculate confusion matrix
cm1 = confusion_matrix(y_moons, tree_clf.predict(X_moons), labels=tree_clf.classes_)
cm2 = confusion_matrix(y_moons, vote_clf.predict(X_moons), labels=vote_clf.classes_)

#display confudion matrix
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm1,display_labels=tree_clf.classes_)
disp1.plot()

disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2,display_labels=vote_clf.classes_)
disp2.plot()

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

print("Scores for single tree")
precison, recall, f1_score, support = precision_recall_fscore_support(y_moons,
                                                                      tree_clf.predict(X_moons),
                                                                      beta=1.0) #beta=1 gives f1 score

print('Precision = ', precison, 'Recall = ', recall, 'F1 score = ', f1_score)

print("Scores for Voting")
precison, recall, f1_score, support = precision_recall_fscore_support(y_moons,
                                                                      vote_clf.predict(X_moons),
                                                                      beta=1.0) #beta=1 gives f1 score

print('Precision = ', precison, 'Recall = ', recall, 'F1 score = ', f1_score)


# Bagging of Decision Trees

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                            max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_moons, y_moons)

In [ ]:
# @title Plotting
# extra code –

def plot_decision_boundary(clf, X, y, alpha=1.0):
    axes=[-1.5, 2.4, -1, 1.5]
    x1, x2 = np.meshgrid(np.linspace(axes[0], axes[1], 100),
                         np.linspace(axes[2], axes[3], 100))
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)

    plt.contourf(x1, x2, y_pred, alpha=0.3, cmap='Wistia')
    plt.contour(x1, x2, y_pred, cmap="Greys", alpha=0.8)
    colors = ["#78785c", "#c47b27"]
    markers = ("o", "^")
    for idx in (0, 1):
        plt.plot(X[:, 0][y == idx], X[:, 1][y == idx],
                 color=colors[idx], marker=markers[idx], linestyle="none")
    plt.axis(axes)
    plt.xlabel(r"$x_1$")
    plt.ylabel(r"$x_2$", rotation=0)

tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_moons, y_moons)

fig, axes = plt.subplots(ncols=2, figsize=(10, 4), sharey=True)
plt.sca(axes[0])
plot_decision_boundary(tree_clf, X_moons, y_moons)
plt.title("Decision Tree")
plt.sca(axes[1])
plot_decision_boundary(bag_clf, X_moons, y_moons)
plt.title("Decision Trees with Bagging")
plt.ylabel("")
plt.show()

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=10,
                                 n_jobs=-1, random_state=42)
rnd_clf.fit(X_moons, y_moons)



In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(10, 4), sharey=True)
plt.sca(axes[0])
plot_decision_boundary(tree_clf, X_moons, y_moons)
plt.title("Decision Tree")
plt.sca(axes[1])
plot_decision_boundary(rnd_clf, X_moons, y_moons)
plt.title("Random Forest")
plt.ylabel("")
plt.show()

### Feature Importances

In [ ]:
rnd_clf.feature_importances_

# Gradient Boosting (from scratch)
Shown here for a regression problem

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3 * X[:, 0] ** 2 + 0.05 * np.random.randn(100)  # y = 3x² + Gaussian noise

# Train a decision tree regression model
tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg1.fit(X, y)

# Now let's train another decision tree regressor on the residual errors made by the previous predictor:
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2, random_state=43)
tree_reg2.fit(X, y2)

# Repeat
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2, random_state=44)
tree_reg3.fit(X, y3)

In [ ]:
# @title Plotting
def plot_predictions(regressors, X, y, axes, style,
                     label=None, data_style="b.", data_label=None):
    x1 = np.linspace(axes[0], axes[1], 500)
    y_pred = sum(regressor.predict(x1.reshape(-1, 1))
                 for regressor in regressors)
    plt.plot(X[:, 0], y, data_style, label=data_label)
    plt.plot(x1, y_pred, style, linewidth=2, label=label)
    if label or data_label:
        plt.legend(loc="upper center")
    plt.axis(axes)

plt.figure(figsize=(11, 11))

plt.subplot(3, 2, 1)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.2, 0.8], style="g-",
                 label="$h_1(x_1)$", data_label="Training set")
plt.ylabel("$y$  ", rotation=0)
plt.title("Residuals and tree predictions")

plt.subplot(3, 2, 2)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.2, 0.8], style="r-",
                 label="$h(x_1) = h_1(x_1)$", data_label="Training set")
plt.title("Ensemble predictions")

plt.subplot(3, 2, 3)
plot_predictions([tree_reg2], X, y2, axes=[-0.5, 0.5, -0.4, 0.6], style="g-",
                 label="$h_2(x_1)$", data_style="k+",
                 data_label="Residuals: $y - h_1(x_1)$")
plt.ylabel("$y$  ", rotation=0)

plt.subplot(3, 2, 4)
plot_predictions([tree_reg1, tree_reg2], X, y, axes=[-0.5, 0.5, -0.2, 0.8],
                  style="r-", label="$h(x_1) = h_1(x_1) + h_2(x_1)$")

plt.subplot(3, 2, 5)
plot_predictions([tree_reg3], X, y3, axes=[-0.5, 0.5, -0.4, 0.6], style="g-",
                 label="$h_3(x_1)$", data_style="k+",
                 data_label="Residuals: $y - h_1(x_1) - h_2(x_1)$")
plt.xlabel("$x_1$")
plt.ylabel("$y$  ", rotation=0)

plt.subplot(3, 2, 6)
plot_predictions([tree_reg1, tree_reg2, tree_reg3], X, y,
                 axes=[-0.5, 0.5, -0.2, 0.8], style="r-",
                 label="$h(x_1) = h_1(x_1) + h_2(x_1) + h_3(x_1)$")
plt.xlabel("$x_1$")

# Gradient Boosting with ScikitLearn

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

#create a gradient boositng regressor
gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3,
                                 learning_rate=1.0, random_state=42)
gbrt.fit(X, y)

#find the optimal gradient boosting regressor by stopping training when increasing
#the number of estimators does not improve performance
gbrt_best = GradientBoostingRegressor(
    max_depth=2, learning_rate=0.05, n_estimators=500,
    n_iter_no_change=10, random_state=42)
gbrt_best.fit(X, y)



In [ ]:
# @title Plotting
fig, axes = plt.subplots(ncols=2, figsize=(10, 4), sharey=True)

plt.sca(axes[0])
plot_predictions([gbrt], X, y, axes=[-0.5, 0.5, -0.1, 0.8], style="r-",
                 label="Ensemble predictions")
plt.title(f"learning_rate={gbrt.learning_rate}, "
          f"n_estimators={gbrt.n_estimators_}")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)

plt.sca(axes[1])
plot_predictions([gbrt_best], X, y, axes=[-0.5, 0.5, -0.1, 0.8], style="r-")
plt.title(f"learning_rate={gbrt_best.learning_rate}, "
          f"n_estimators={gbrt_best.n_estimators_}")
plt.xlabel("$x_1$")